# Ternary NoProp Validation on Colab Free Tier

This notebook is a practical Wave 0/1/2 harness for validating **NoProp-DT x ternary weights** under free-tier constraints.

## What this notebook does
- Runs Wave 0 baselines (FP32 NoProp, FP32 backprop, ternary backprop).
- Runs Wave 1 existence tests (STE, Trust, DQT-style, HESTIA-style).
- Logs validation perplexity, dead-neuron rate, gradient norms, and memory usage.
- Saves resumable checkpoints and CSV logs to Google Drive.
- Provides a free-tier alternatives report (Kaggle, Studio Lab, Lightning Free).

## Colab free-tier reality
- Runtime and GPU availability are dynamic and not guaranteed.
- Free runtimes can terminate early; this notebook is checkpoint-first.
- Design target is a small transformer and WikiText-2 for fast iteration.

In [ ]:
# Colab setup
import os, sys, json, time, random, math, platform
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

!pip -q install -U torch datasets transformers pandas matplotlib tqdm

In [ ]:
# Paths, seed, and hardware probe
import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

RUN_NAME = time.strftime('noprop_ternary_%Y%m%d_%H%M%S')
BASE_DIR = Path('/content')
OUT_DIR = Path('/content/drive/MyDrive/drex_noprop_ternary') if IN_COLAB else Path('/tmp/drex_noprop_ternary')
RUN_DIR = OUT_DIR / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = RUN_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM (GB):', round(props.total_memory / 1024**3, 2))
print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('Run dir:', RUN_DIR)

In [ ]:
# Dataset: WikiText-2, tokenized to fixed-length LM segments
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_TOKENIZER = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(MODEL_TOKENIZER)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

class LMDataset(Dataset):
    def __init__(self, token_ids, seq_len=128):
        self.seq_len = seq_len
        self.tokens = token_ids
        self.n = max(0, (len(self.tokens) - 1) // self.seq_len)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        i = idx * self.seq_len
        x = self.tokens[i:i+self.seq_len]
        y = self.tokens[i+1:i+self.seq_len+1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

def build_wikitext2(seq_len=128, train_chars=2000000, val_chars=200000, batch_size=16):
    ds_train = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    ds_val = load_dataset('wikitext', 'wikitext-2-raw-v1', split='validation')

    txt_train = '\n'.join(ds_train['text'])[:train_chars]
    txt_val = '\n'.join(ds_val['text'])[:val_chars]

    tok_train = tokenizer(txt_train, add_special_tokens=False)['input_ids']
    tok_val = tokenizer(txt_val, add_special_tokens=False)['input_ids']

    train_ds = LMDataset(tok_train, seq_len=seq_len)
    val_ds = LMDataset(tok_val, seq_len=seq_len)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader

train_loader, val_loader = build_wikitext2(seq_len=128, batch_size=16)
print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))

In [ ]:
# Quantization primitives (STE, Trust-style, DQT-style, HESTIA-style)
import torch.nn as nn
import torch.nn.functional as F

class TernarySTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w, delta):
        ctx.save_for_backward(w, delta)
        return torch.where(w > delta, torch.ones_like(w), torch.where(w < -delta, -torch.ones_like(w), torch.zeros_like(w)))

    @staticmethod
    def backward(ctx, grad_output):
        w, delta = ctx.saved_tensors
        mask = (w.abs() <= 1.5 * delta).to(grad_output.dtype)
        return grad_output * mask, None

def ternary_ste(w, delta):
    return TernarySTE.apply(w, delta)

def trust_scale(w, delta):
    d = (w.abs() - delta).abs()
    return (1.0 / (1.0 + d)).clamp(0.05, 1.0)

def stochastic_ternary_project(w, delta):
    a = w.abs() / (delta + 1e-8)
    p = a.clamp(0.0, 1.0)
    keep = (torch.rand_like(w) < p).float()
    signs = w.sign()
    return keep * signs

class BitLinear(nn.Module):
    def __init__(self, in_f, out_f, mode='fp32', bias=True):
        super().__init__()
        self.mode = mode

        if mode == 'hestia':
            self.logits = nn.Parameter(torch.randn(out_f, in_f, 3) * 0.02)
            self.bias = nn.Parameter(torch.zeros(out_f)) if bias else None
        else:
            self.weight = nn.Parameter(torch.randn(out_f, in_f) * 0.02)
            self.bias = nn.Parameter(torch.zeros(out_f)) if bias else None

    def _delta(self, w):
        return 0.7 * w.abs().mean().detach() + 1e-8

    def forward(self, x, tau=1.0):
        if self.mode == 'fp32':
            wq = self.weight
        elif self.mode == 'ste':
            d = self._delta(self.weight)
            wq = ternary_ste(self.weight, d)
        elif self.mode == 'trust':
            d = self._delta(self.weight)
            wq = ternary_ste(self.weight, d)
            wq = wq * trust_scale(self.weight, d)
        elif self.mode == 'dqt':
            d = self._delta(self.weight)
            wq = stochastic_ternary_project(self.weight, d)
        elif self.mode == 'hestia':
            states = torch.tensor([-1.0, 0.0, 1.0], device=x.device).view(1, 1, 3)
            probs = F.softmax(self.logits / max(tau, 1e-3), dim=-1)
            wq = (probs * states).sum(dim=-1)
        else:
            raise ValueError(f'Unknown mode: {self.mode}')
        return F.linear(x, wq, self.bias)

In [ ]:
# Small transformer blocks + NoProp-compatible blockwise interface
class TinyBlock(nn.Module):
    def __init__(self, d_model, n_heads, mode='fp32'):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ff1 = BitLinear(d_model, 4 * d_model, mode=mode)
        self.ff2 = BitLinear(4 * d_model, d_model, mode=mode)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x, tau=1.0):
        T = x.size(1)
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        a, _ = self.attn(x, x, x, attn_mask=mask)
        x = self.ln1(x + a)
        h = F.gelu(self.ff1(x, tau=tau))
        h = self.ff2(h, tau=tau)
        x = self.ln2(x + h)
        return x

class TinyNoPropTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_blocks=6, mode='fp32', max_len=256):
        super().__init__()
        self.mode = mode
        self.n_blocks = n_blocks
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([TinyBlock(d_model, n_heads, mode=mode) for _ in range(n_blocks)])
        self.noise_proj = nn.ModuleList([BitLinear(d_model, d_model, mode='fp32') for _ in range(n_blocks)])
        self.head = BitLinear(d_model, vocab_size, mode='fp32')

    def base_embed(self, x):
        B, T = x.shape
        p = torch.arange(T, device=x.device).unsqueeze(0).expand(B, T)
        return self.tok(x) + self.pos(p)

    def forward_global(self, x, tau=1.0):
        h = self.base_embed(x)
        for blk in self.blocks:
            h = blk(h, tau=tau)
        return self.head(h)

    def forward_block_local(self, block_idx, x, z_noisy, tau=1.0):
        h = self.base_embed(x).detach()
        h = h + self.noise_proj[block_idx](z_noisy.detach())
        out = self.blocks[block_idx](h, tau=tau)
        logits = self.head(out)
        return out, logits

def dead_neuron_rate(t):
    return (t.abs() < 1e-6).float().mean().item()

In [ ]:
# Training and validation utilities
def evaluate_ppl(model, val_loader, max_batches=50):
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (x, y) in enumerate(val_loader):
            if i >= max_batches:
                break
            x, y = x.to(device), y.to(device)
            logits = model.forward_global(x)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            losses.append(loss.item())
    model.train()
    m = float(sum(losses) / max(1, len(losses)))
    return m, float(math.exp(min(m, 20.0)))

def get_grad_norm(module):
    sq = 0.0
    for p in module.parameters():
        if p.grad is not None:
            sq += float((p.grad.detach() ** 2).sum().item())
    return math.sqrt(max(sq, 0.0))

def save_ckpt(model, opt, step, tag):
    path = CKPT_DIR / f'{tag}_step_{step:06d}.pt'
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'step': step}, path)
    return str(path)

In [ ]:
# Experiment runners: global backprop and NoProp local denoising
def run_global(exp_id, mode='fp32', steps=1000, lr=3e-4, eval_every=200):
    vocab = len(tokenizer)
    model = TinyNoPropTransformer(vocab_size=vocab, mode=mode).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    logs = []
    it = iter(train_loader)
    tau = 1.5

    for step in range(1, steps + 1):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_loader)
            x, y = next(it)
        x, y = x.to(device), y.to(device)

        opt.zero_grad(set_to_none=True)
        logits = model.forward_global(x, tau=tau)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        loss.backward()
        gn = get_grad_norm(model)
        opt.step()

        if mode == 'hestia':
            tau = max(0.05, tau * 0.999)

        if step % eval_every == 0 or step == 1:
            val_loss, val_ppl = evaluate_ppl(model, val_loader)
            mem = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
            logs.append({
                'exp_id': exp_id, 'step': step, 'train_loss': float(loss.item()),
                'val_loss': val_loss, 'val_ppl': val_ppl, 'grad_norm': gn,
                'dead_rate': 0.0, 'max_mem_gb': mem
            })

    ckpt = save_ckpt(model, opt, steps, exp_id)
    return model, pd.DataFrame(logs), ckpt

def run_noprop(exp_id, mode='ste', steps=1000, lr=3e-4, eval_every=200):
    vocab = len(tokenizer)
    model = TinyNoPropTransformer(vocab_size=vocab, mode=mode, n_blocks=6).to(device)

    block_opts = []
    shared_params = list(model.tok.parameters()) + list(model.pos.parameters()) + list(model.head.parameters())
    for b in range(model.n_blocks):
        params = list(model.blocks[b].parameters()) + list(model.noise_proj[b].parameters()) + shared_params
        block_opts.append(torch.optim.AdamW(params, lr=lr))

    logs = []
    it = iter(train_loader)
    tau = 1.5
    noise_sched = torch.linspace(1.0, 0.1, model.n_blocks, device=device)

    for step in range(1, steps + 1):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_loader)
            x, y = next(it)
        x, y = x.to(device), y.to(device)

        y_emb = model.tok(y).detach()
        block_grad_norms = []
        block_dead_rates = []
        block_losses = []

        for b in range(model.n_blocks):
            opt = block_opts[b]
            opt.zero_grad(set_to_none=True)

            sigma = noise_sched[b]
            z_noisy = y_emb + sigma * torch.randn_like(y_emb)
            z_pred, logits = model.forward_block_local(b, x, z_noisy, tau=tau)

            l2 = F.mse_loss(z_pred, y_emb)
            ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            alpha_t = 1.0 + 0.1 * (b / max(1, model.n_blocks - 1))
            loss = alpha_t * l2 + 0.1 * ce
            loss.backward()

            block_grad_norms.append(get_grad_norm(model.blocks[b]))
            block_dead_rates.append(dead_neuron_rate(z_pred.detach()))
            block_losses.append(float(loss.item()))
            opt.step()

        if mode == 'hestia':
            tau = max(0.05, tau * 0.999)

        if step % eval_every == 0 or step == 1:
            val_loss, val_ppl = evaluate_ppl(model, val_loader)
            mem = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
            logs.append({
                'exp_id': exp_id, 'step': step,
                'train_loss': float(sum(block_losses) / len(block_losses)),
                'val_loss': val_loss, 'val_ppl': val_ppl,
                'grad_norm': float(sum(block_grad_norms) / len(block_grad_norms)),
                'dead_rate': float(sum(block_dead_rates) / len(block_dead_rates)),
                'max_mem_gb': mem
            })

    ckpt = save_ckpt(model, block_opts[0], steps, exp_id)
    return model, pd.DataFrame(logs), ckpt

In [ ]:
# Wave matrix (free-tier budget defaults)
EXPERIMENTS = [
    {'id': '0A_fp32_noprop', 'kind': 'noprop', 'mode': 'fp32', 'steps': 800},
    {'id': '0B_ternary_backprop', 'kind': 'global', 'mode': 'trust', 'steps': 800},
    {'id': '0C_fp32_backprop', 'kind': 'global', 'mode': 'fp32', 'steps': 800},
    {'id': '1A_noprop_ste', 'kind': 'noprop', 'mode': 'ste', 'steps': 800},
    {'id': '1B_noprop_trust', 'kind': 'noprop', 'mode': 'trust', 'steps': 800},
    {'id': '1C_noprop_dqt', 'kind': 'noprop', 'mode': 'dqt', 'steps': 800},
    {'id': '1D_noprop_hestia', 'kind': 'noprop', 'mode': 'hestia', 'steps': 800}
]
EXPERIMENTS

In [ ]:
# Run selected experiments
RUN_IDS = [e['id'] for e in EXPERIMENTS]

all_logs = []
results = []

for e in EXPERIMENTS:
    if e['id'] not in RUN_IDS:
        continue
    print(f"Running {e['id']} ({e['kind']}, {e['mode']})")
    t0 = time.time()
    if e['kind'] == 'global':
        model, logs_df, ckpt = run_global(e['id'], mode=e['mode'], steps=e['steps'])
    else:
        model, logs_df, ckpt = run_noprop(e['id'], mode=e['mode'], steps=e['steps'])
    elapsed = time.time() - t0

    logs_path = RUN_DIR / f"{e['id']}_logs.csv"
    logs_df.to_csv(logs_path, index=False)
    all_logs.append(logs_df)

    last = logs_df.iloc[-1].to_dict()
    last['elapsed_sec'] = elapsed
    last['checkpoint'] = ckpt
    results.append(last)
    print(f"  done in {elapsed/60:.1f} min | val_ppl={last['val_ppl']:.2f} | dead={last['dead_rate']:.3f}")

summary_df = pd.DataFrame(results).sort_values('val_ppl')
summary_path = RUN_DIR / 'summary.csv'
summary_df.to_csv(summary_path, index=False)
summary_df

In [ ]:
# Plots and quick pass/fail checks
if all_logs:
    df = pd.concat(all_logs, ignore_index=True)
    fig, ax = plt.subplots(1, 3, figsize=(18, 4))
    for exp_id, g in df.groupby('exp_id'):
        ax[0].plot(g['step'], g['val_ppl'], label=exp_id)
        ax[1].plot(g['step'], g['dead_rate'], label=exp_id)
        ax[2].plot(g['step'], g['grad_norm'], label=exp_id)
    ax[0].set_title('Validation Perplexity')
    ax[1].set_title('Dead Neuron Rate')
    ax[2].set_title('Gradient Norm')
    for a in ax:
        a.set_xlabel('step')
        a.legend(fontsize=7)
        a.grid(True, alpha=0.3)
    plt.show()

    gate = []
    for exp_id in ['1A_noprop_ste', '1B_noprop_trust', '1C_noprop_dqt', '1D_noprop_hestia']:
        g = df[df.exp_id == exp_id]
        if len(g) == 0:
            continue
        dec = g['train_loss'].iloc[-1] < g['train_loss'].iloc[0]
        dead_ok = g['dead_rate'].iloc[-1] < 0.30
        gate.append({'exp_id': exp_id, 'loss_down': bool(dec), 'dead_under_30pct': bool(dead_ok)})
    pd.DataFrame(gate)

## Wave-2 diagnostics checklist

Run these after at least one Wave-1 variant shows stable loss decrease:

1. Gradient ratio test: log NoProp block grad norms vs 0C backprop grad norms at steps 1k, 5k, 10k, 50k.
2. Dead-zone mapping: histogram ternary states and latent values over time (trimodal vs zero-peaked collapse).
3. Block depth sweep: re-run best variant at T=5, 10, 20 and compare quality/time.
4. Noise schedule sweep: linear vs cosine vs learnable.
5. Gradient amplification: enable per-block alpha_t and test if dead-rate decreases with better perplexity.

For publication-quality evidence, increase each run to >= 50k steps and keep identical seed/data budgets across methods.

## Alternative free-tier options (research report)

### 1) Google Colab Free
- Best for: fastest startup and low-friction notebook iteration.
- Constraints: dynamic limits, no guaranteed GPU type, free sessions can terminate; max run duration is typically up to 12 hours depending on usage and availability.
- Why still first choice: easiest Drive integration and broad community examples for LM prototyping.

Source: Colab FAQ (resource limits and runtime duration)
- https://research.google.com/colaboratory/faq.html

### 2) Kaggle Notebooks (Free)
- Best for: stable reproducible notebook runs with explicit environment metadata and free GPU/TPU options.
- Notable published limits: up to 12h CPU/GPU session execution, 20GB saved working disk, idle timeout behavior.
- Practical fit: strong fallback when Colab GPU is unavailable; good for long queued batch runs and artifact sharing.

Source: Kaggle Notebooks docs (technical specifications and accelerator options)
- https://www.kaggle.com/docs/notebooks

### 3) Amazon SageMaker Studio Lab (Free)
- Best for: no-credit-card free Jupyter environment with CPU/GPU choices and persistent storage.
- Published baseline: 15GB persistent storage, browser JupyterLab workflow.
- Practical fit: good secondary fallback for checkpoint-heavy iterative experiments.

Source: Studio Lab home/FAQ links
- https://studiolab.sagemaker.aws/

### 4) Lightning AI Free
- Best for: IDE + cloud studio workflow, useful if you want local IDE attachment and quick switch between CPU/GPU.
- Published free plan highlights: monthly credits, 1 active free CPU studio, free-tier session restart constraints for long-running sessions.
- Practical fit: useful for experimentation pipelines; less universally available in academia than Colab/Kaggle.

Source: Lightning pricing page
- https://lightning.ai/pricing

### Recommended fallback order for this project
1. Colab Free (primary)
2. Kaggle Notebooks (first fallback)
3. Studio Lab (second fallback)
4. Lightning Free (workflow-oriented fallback)

In [ ]:
# Persist a machine-readable run summary
report = {
    'run_name': RUN_NAME,
    'device': str(device),
    'seed': SEED,
    'run_dir': str(RUN_DIR),
    'experiments': results if 'results' in globals() else []
}
with open(RUN_DIR / 'run_report.json', 'w') as f:
    json.dump(report, f, indent=2)
print('Saved:', RUN_DIR / 'run_report.json')